# Validación de ECU911: volumen de emergencias a t+3
Comparamos el bosque ganador y la red neuronal de Semana 11, sesión 1, con sus mismos hiperparámetros. Solo usamos el bloque de entrenamiento original: el período desde 2025-08-17 queda fuera de esta comparación. Tres pliegues por esquema permiten medir variación sin multiplicar innecesariamente el entrenamiento.

La validación temporal separa fechas completas y purga tres días antes de cada validación: toda etiqueta de entrenamiento a t+3 ocurre antes del primer día validado. El esquema aleatorio por filas es una comparación deliberadamente optimista, pues mezcla fechas, cantones e información futura.

Hipótesis de costo para planificación (sin tarifa monetaria validada): subestimar una emergencia pesa tres veces más que sobreestimarla, por el riesgo de capacidad insuficiente. Medimos `media(3 * max(real - predicho, 0) + max(predicho - real, 0))`, además del MAE. No representa dólares ni sustituye evaluar las alertas de demanda alta del proyecto final.


In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from threadpoolctl import threadpool_limits
from pipeline_proyecto import RUTA_DATOS, cargar_y_dividir, construir_preparacion

X, X_reservado, y, y_reservado = cargar_y_dividir()
fechas = pd.read_csv(RUTA_DATOS, usecols=["fecha"], parse_dates=["fecha"]).loc[X.index, "fecha"].reset_index(drop=True)
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
fechas_unicas = np.sort(fechas.unique())
pliegues_temporales = []
for ent_dias, val_dias in TimeSeriesSplit(n_splits=3, gap=3).split(fechas_unicas):
    ent = np.flatnonzero(fechas.isin(fechas_unicas[ent_dias]))
    val = np.flatnonzero(fechas.isin(fechas_unicas[val_dias]))
    assert fechas.iloc[ent].max() + pd.Timedelta(days=3) < fechas.iloc[val].min()
    pliegues_temporales.append((ent, val))
esquemas = {"temporal": pliegues_temporales, "aleatorio": list(KFold(n_splits=3, shuffle=True, random_state=42).split(X))}
modelos = {
    "bosque regularizado": RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=20, random_state=42, n_jobs=2),
    "red neuronal": TransformedTargetRegressor(regressor=MLPRegressor(hidden_layer_sizes=(64,), max_iter=300, batch_size=1024, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15, random_state=42), transformer=StandardScaler()),
}
print("Filas para CV:", len(X), "Período reservado:", len(X_reservado))


Filas para CV: 337120 Período reservado: 84448


In [2]:
filas = []
for esquema, pliegues in esquemas.items():
    for nombre, modelo in modelos.items():
        for numero, (ent, val) in enumerate(pliegues, 1):
            pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", clone(modelo))])
            inicio = time.perf_counter()
            with threadpool_limits(limits=2):
                pipeline.fit(X.iloc[ent], y.iloc[ent])
                pred = pipeline.predict(X.iloc[val])
            error = y.iloc[val].to_numpy() - pred
            fila = dict(esquema=esquema, modelo=nombre, pliegue=numero,
                n_ent=len(ent), n_val=len(val), fin_ent=fechas.iloc[ent].max().date().isoformat(),
                inicio_val=fechas.iloc[val].min().date().isoformat(), fin_val=fechas.iloc[val].max().date().isoformat(),
                mae=mean_absolute_error(y.iloc[val], pred), costo_asimetrico=np.mean(3*np.maximum(error, 0)+np.maximum(-error, 0)),
                subestimacion_media=np.maximum(error, 0).mean(), sobreestimacion_media=np.maximum(-error, 0).mean(), segundos=time.perf_counter()-inicio)
            filas.append(fila)
            pd.DataFrame(filas).to_csv("validacion_proyecto_pliegues.csv", index=False)
            print(fila, flush=True)


{'esquema': 'temporal', 'modelo': 'bosque regularizado', 'pliegue': 1, 'n_ent': 83776, 'n_val': 84224, 'fin_ent': '2022-07-09', 'inicio_val': '2022-07-13', 'fin_val': '2023-07-23', 'mae': 6.831535494348732, 'costo_asimetrico': 12.44946994007085, 'subestimacion_media': 2.80896722286106, 'sobreestimacion_media': 4.022568271487671, 'segundos': 18.33790363301523}


{'esquema': 'temporal', 'modelo': 'bosque regularizado', 'pliegue': 2, 'n_ent': 168000, 'n_val': 84224, 'fin_ent': '2023-07-20', 'inicio_val': '2023-07-24', 'fin_val': '2024-08-02', 'mae': 5.944793428416323, 'costo_asimetrico': 11.412308744511206, 'subestimacion_media': 2.733757658047442, 'sobreestimacion_media': 3.2110357703688797, 'segundos': 45.90613089501858}


{'esquema': 'temporal', 'modelo': 'bosque regularizado', 'pliegue': 3, 'n_ent': 252224, 'n_val': 84224, 'fin_ent': '2024-07-30', 'inicio_val': '2024-08-03', 'fin_val': '2025-08-13', 'mae': 5.768648240157493, 'costo_asimetrico': 11.023557470590276, 'subestimacion_media': 2.627454615216392, 'sobreestimacion_media': 3.141193624941102, 'segundos': 68.3390200319991}


{'esquema': 'temporal', 'modelo': 'red neuronal', 'pliegue': 1, 'n_ent': 83776, 'n_val': 84224, 'fin_ent': '2022-07-09', 'inicio_val': '2022-07-13', 'fin_val': '2023-07-23', 'mae': 7.20606517260418, 'costo_asimetrico': 12.266020464414916, 'subestimacion_media': 2.5299776459053684, 'sobreestimacion_media': 4.676087526698811, 'segundos': 1.5124532090267166}


{'esquema': 'temporal', 'modelo': 'red neuronal', 'pliegue': 2, 'n_ent': 168000, 'n_val': 84224, 'fin_ent': '2023-07-20', 'inicio_val': '2023-07-24', 'fin_val': '2024-08-02', 'mae': 6.650701369831557, 'costo_asimetrico': 10.868932459331248, 'subestimacion_media': 2.1091155447498453, 'sobreestimacion_media': 4.541585825081713, 'segundos': 8.76713800500147}


{'esquema': 'temporal', 'modelo': 'red neuronal', 'pliegue': 3, 'n_ent': 252224, 'n_val': 84224, 'fin_ent': '2024-07-30', 'inicio_val': '2024-08-03', 'fin_val': '2025-08-13', 'mae': 6.282850267054209, 'costo_asimetrico': 11.893590445470894, 'subestimacion_media': 2.8053700892083433, 'sobreestimacion_media': 3.4774801778458646, 'segundos': 10.128284421021817}


... (2 outputs left out)

{'esquema': 'aleatorio', 'modelo': 'bosque regularizado', 'pliegue': 3, 'n_ent': 224747, 'n_val': 112373, 'fin_ent': '2025-08-13', 'inicio_val': '2021-07-01', 'fin_val': '2025-08-13', 'mae': 5.979835826544084, 'costo_asimetrico': 11.794919990348959, 'subestimacion_media': 2.9075420819024376, 'sobreestimacion_media': 3.072293744641647, 'segundos': 59.61996423502569}


{'esquema': 'aleatorio', 'modelo': 'red neuronal', 'pliegue': 1, 'n_ent': 224746, 'n_val': 112374, 'fin_ent': '2025-08-13', 'inicio_val': '2021-07-01', 'fin_val': '2025-08-13', 'mae': 6.2588375218579095, 'costo_asimetrico': 12.216693642207732, 'subestimacion_media': 2.9789280601749115, 'sobreestimacion_media': 3.2799094616829985, 'segundos': 4.750154793000547}


{'esquema': 'aleatorio', 'modelo': 'red neuronal', 'pliegue': 2, 'n_ent': 224747, 'n_val': 112373, 'fin_ent': '2025-08-13', 'inicio_val': '2021-07-01', 'fin_val': '2025-08-13', 'mae': 6.216128879856149, 'costo_asimetrico': 12.744531724831418, 'subestimacion_media': 3.264201422487636, 'sobreestimacion_media': 2.9519274573685133, 'segundos': 13.546711172995856}


{'esquema': 'aleatorio', 'modelo': 'red neuronal', 'pliegue': 3, 'n_ent': 224747, 'n_val': 112373, 'fin_ent': '2025-08-13', 'inicio_val': '2021-07-01', 'fin_val': '2025-08-13', 'mae': 6.603915156608888, 'costo_asimetrico': 11.761254461892765, 'subestimacion_media': 2.578669652641937, 'sobreestimacion_media': 4.025245503966952, 'segundos': 5.380203626991715}


In [3]:
pliegues = pd.DataFrame(filas)
# Desviación poblacional de los tres errores observados (ddof=0).
resumen = pliegues.groupby(["esquema", "modelo"]).agg(
    mae_promedio=("mae", "mean"), mae_desviacion=("mae", lambda x: x.std(ddof=0)),
    mae_peor=("mae", "max"), costo_promedio=("costo_asimetrico", "mean"),
    costo_desviacion=("costo_asimetrico", lambda x: x.std(ddof=0))).reset_index()
resumen.to_csv("validacion_proyecto_resumen.csv", index=False)
print(resumen.to_string(index=False))
for esquema in esquemas:
    print("MAE por pliegue:", esquema)
    print(pliegues[pliegues.esquema == esquema].pivot(index="pliegue", columns="modelo", values="mae").to_string())
assert len(pliegues) == 12


  esquema              modelo  mae_promedio  mae_desviacion  mae_peor  costo_promedio  costo_desviacion
aleatorio bosque regularizado      5.877286        0.085510  5.979836       11.763191          0.187278
aleatorio        red neuronal      6.359627        0.173615  6.603915       12.240827          0.401784
 temporal bosque regularizado      6.181659        0.465125  6.831535       11.628445          0.601854
 temporal        red neuronal      6.713206        0.379483  7.206065       11.676181          0.590714
MAE por pliegue: temporal
modelo   bosque regularizado  red neuronal
pliegue                                   
1                   6.831535      7.206065
2                   5.944793      6.650701
3                   5.768648      6.282850
MAE por pliegue: aleatorio
modelo   bosque regularizado  red neuronal
pliegue                                   
1                   5.881515      6.258838
2                   5.770508      6.216129
3                   5.979836      6.6039

Los pliegues temporales tienen entrenamiento creciente: su variación refleja cambios de período y cantidad de datos. La red usa parada temprana con una partición aleatoria interna de entrenamiento, igual que en la sesión anterior; nunca usa la validación externa para detenerse. La desviación entre tres pliegues no es un intervalo de confianza ni una prueba de significancia. Se prioriza el esquema temporal para la decisión de despliegue; el aleatorio sirve para mostrar el efecto del esquema.
